# Train Helpful and Harmless GPT-2 LoRA Adapters

This notebook is a Google Colab runner for training two lightweight prototype adapters from the same fresh GPT-2 base model:

- Helpful adapter from `Anthropic/hh-rlhf`, `helpful-base`
- Harmless adapter from `Anthropic/hh-rlhf`, `harmless-base`

Training uses the `chosen` responses for supervised causal language modeling. The `rejected` responses are not used yet. This is not full RLHF or PPO, and it is not the final preference-aware coefficient correction method $\lambda = f(p, R)$.

## Check GPU

In Colab, select **Runtime > Change runtime type > T4 GPU** before training.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Training on CPU will be much slower.")

## Clone or update repository

This cell always starts from `/content`. It updates an existing clone or creates a new one, avoiding nested folders such as `/content/master-thesis/master-thesis`.

In [ ]:
%cd /content

from pathlib import Path
import subprocess

repo_path = Path("/content/master-thesis")

if repo_path.exists():
    print("Repository already exists. Pulling the latest changes...")
    subprocess.run(["git", "-C", str(repo_path), "pull"], check=True)
else:
    print("Cloning the repository...")
    subprocess.run(
        ["git", "clone", "https://github.com/NZhang137/master-thesis.git"],
        check=True,
    )

%cd /content/master-thesis

Confirm that the repository root contains `README.md`, `scripts/`, `src/`, `notebooks/`, and `results/`.

In [ ]:
!pwd
!ls
!ls scripts

## Install dependencies

`torchao` is not needed for this GPT-2 + PEFT prototype. Removing an old preinstalled version avoids compatibility errors.

In [ ]:
!pip uninstall -y torchao
!pip install -q -U transformers datasets peft accelerate

If `torchao` was imported earlier in this Colab session, select **Runtime > Restart session**. Then rerun the repository and installation cells before continuing.

## Train helpful and harmless adapters

The script first trains the Helpful adapter. It then releases that model, reloads a fresh GPT-2 base model with a new LoRA adapter, and trains the Harmless adapter.

The default small prototype run uses 100 examples per objective and 2 epochs. It may take several minutes on a Colab GPU.

In [ ]:
!python scripts/train_hh_rlhf_adapters.py --split "train[:100]" --num_epochs 2

## Check adapter files

The checker verifies that both output folders contain a PEFT configuration and adapter weights.

In [ ]:
!python scripts/check_adapters.py
!ls adapters

## Zip adapters for local backup

**Warning:** `adapters.zip` is only a local backup for downloading from Colab. The `adapters/` directory and `adapters.zip` must not be committed to GitHub. Generated adapter weights are ignored by the repository's Git rules.

In [ ]:
!zip -r adapters.zip adapters/

Download `adapters.zip` from the Colab file browser if you want to keep a local backup. Do not add the ZIP or adapter-weight files to GitHub.

## Git safety check

The status output should not list `adapters/`, `adapters.zip`, `.safetensors`, `.bin`, checkpoints, or model files. These generated artifacts must remain local.

In [ ]:
!git status